In [1]:

import requests
import io
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, precision_score, recall_score


# ================================================================
# LIVE REST API ENDPOINT
# ================================================================

HOST_TELEMETRY_API_ENDPOINT = (
    "https://raw.githubusercontent.com/"
    "mwaskom/seaborn-data/master/iris.csv"
)


# ================================================================
# STUDENT TASK — API FETCHING FUNCTION
# ================================================================

def fetch_telemetry_data(url: str) -> pd.DataFrame:
    """
    Fetch CSV telemetry data from an HTTP endpoint.

    Steps:
    1. Send HTTP GET request.
    2. Raise an error if HTTP request fails.
    3. Convert response text into a file-like object.
    4. Read CSV into Pandas DataFrame.
    5. Handle network/request errors.
    """

    try:
        response = requests.get(url, timeout=30)

        # Raise exception for HTTP errors such as 404, 500, etc.
        response.raise_for_status()

        # Convert HTTP response text into file-like object
        csv_data = io.StringIO(response.text)

        # Parse CSV into DataFrame
        df = pd.read_csv(csv_data)

        return df

    except requests.exceptions.RequestException as error:
        print("API request failed:", error)
        raise

In [2]:

# ================================================================
# TASK 1 — REST API INGESTION & TARGET TRANSFORMATION
# ================================================================

df = fetch_telemetry_data(HOST_TELEMETRY_API_ENDPOINT)


# Rename Iris columns to cloud telemetry terminology
df = df.rename(
    columns={
        "sepal_length": "cpu_usage_pct",
        "sepal_width": "memory_usage_pct",
        "petal_length": "disk_io_rate",
        "petal_width": "network_latency_ms"
    }
)


# Convert species into binary failure target
# setosa     -> 0 = Normal
# versicolor -> 1 = Failure
# virginica  -> 1 = Failure

df["Is_Failure"] = (
    df["species"] != "setosa"
).astype(int)


# Remove original categorical species column
df = df.drop(columns=["species"])


print("\n[TASK 1 SUCCESS] Telemetry API Data Ingested")
print(f"Records: {len(df)}")

print("\nTarget Distribution (0: Normal / 1: Failure):")
print(df["Is_Failure"].value_counts().sort_index())

print("\nSample Feature Table:")
print(df.head(3))



[TASK 1 SUCCESS] Telemetry API Data Ingested
Records: 150

Target Distribution (0: Normal / 1: Failure):
Is_Failure
0     50
1    100
Name: count, dtype: int64

Sample Feature Table:
   cpu_usage_pct  memory_usage_pct  disk_io_rate  network_latency_ms  \
0            5.1               3.5           1.4                 0.2   
1            4.9               3.0           1.4                 0.2   
2            4.7               3.2           1.3                 0.2   

   Is_Failure  
0           0  
1           0  
2           0  


In [3]:
# ================================================================
# TASK 2 — STRATIFIED DATA PARTITIONING
# ================================================================

# Predictive features
X = df[
    [
        "cpu_usage_pct",
        "memory_usage_pct",
        "disk_io_rate",
        "network_latency_ms"
    ]
]

# Target
y = df["Is_Failure"]


# 80/20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


print("\n[TASK 2 SUCCESS] Data Stratified & Split.")

print(
    f"Train Matrix Shape : {X_train.shape} | "
    f"Target Counts: "
    f"0 -> {(y_train == 0).sum()}, "
    f"1 -> {(y_train == 1).sum()}"
)

print(
    f"Test Matrix Shape  : {X_test.shape} | "
    f"Target Counts: "
    f"0 -> {(y_test == 0).sum()}, "
    f"1 -> {(y_test == 1).sum()}"
)



[TASK 2 SUCCESS] Data Stratified & Split.
Train Matrix Shape : (120, 4) | Target Counts: 0 -> 40, 1 -> 80
Test Matrix Shape  : (30, 4) | Target Counts: 0 -> 10, 1 -> 20


In [4]:

baseline_model = DecisionTreeClassifier(
    criterion="gini",
    random_state=42
)

# Train
baseline_model.fit(X_train, y_train)


# Predictions
train_predictions = baseline_model.predict(X_train)
test_predictions = baseline_model.predict(X_test)


# Accuracy
train_accuracy = accuracy_score(
    y_train,
    train_predictions
)

test_accuracy = accuracy_score(
    y_test,
    test_predictions
)


# Overfitting gap
overfitting_gap = (
    train_accuracy - test_accuracy
)


print("\n[TASK 3 SUCCESS] Unpruned Decision Tree Trained.")

print(
    f"Training Accuracy : "
    f"{train_accuracy * 100:.2f}%"
)

print(
    f"Testing Accuracy  : "
    f"{test_accuracy * 100:.2f}%"
)

print(
    f"Overfitting Gap   : "
    f"{overfitting_gap * 100:.2f}%"
)



[TASK 3 SUCCESS] Unpruned Decision Tree Trained.
Training Accuracy : 100.00%
Testing Accuracy  : 100.00%
Overfitting Gap   : 0.00%


In [5]:
# ================================================================
# TASK 4 — GINI FEATURE IMPORTANCE
# ================================================================

feature_importances = baseline_model.feature_importances_


importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": feature_importances
})


# Sort from highest importance to lowest
importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)


print("\n[TASK 4 SUCCESS] Gini Feature Importances Extracted:")

for _, row in importance_df.iterrows():

    print(
        f"- {row['Feature']} : "
        f"{row['Importance']:.4f}"
    )


# ================================================================
# DECISION TREE RULE EXTRACTION
# ================================================================

tree_rules = export_text(
    baseline_model,
    feature_names=list(X.columns)
)


print("\nExtracted Tree Decision Structure:")
print(tree_rules)



[TASK 4 SUCCESS] Gini Feature Importances Extracted:
- disk_io_rate : 1.0000
- cpu_usage_pct : 0.0000
- memory_usage_pct : 0.0000
- network_latency_ms : 0.0000

Extracted Tree Decision Structure:
|--- disk_io_rate <= 2.45
|   |--- class: 0
|--- disk_io_rate >  2.45
|   |--- class: 1



In [6]:
pruned_model = DecisionTreeClassifier(
    criterion="gini",
    max_depth=2,
    min_samples_leaf=5,
    random_state=42
)


# Train pruned tree
pruned_model.fit(
    X_train,
    y_train
)


# Predictions
pruned_train_predictions = pruned_model.predict(
    X_train
)

pruned_test_predictions = pruned_model.predict(
    X_test
)


# Accuracy
pruned_train_accuracy = accuracy_score(
    y_train,
    pruned_train_predictions
)

pruned_test_accuracy = accuracy_score(
    y_test,
    pruned_test_predictions
)


# Gap
pruned_gap = (
    pruned_train_accuracy -
    pruned_test_accuracy
)


print(
    "\n[TASK 5 SUCCESS] "
    "Pruned Decision Tree Trained (max_depth=2)."
)

print(
    f"Pruned Train Accuracy : "
    f"{pruned_train_accuracy * 100:.2f}%"
)

print(
    f"Pruned Test Accuracy  : "
    f"{pruned_test_accuracy * 100:.2f}%"
)

print(
    f"Pruned Overfitting Gap: "
    f"{pruned_gap * 100:.2f}%"
)


if pruned_gap == 0:
    print(
        "Variance Status       : "
        "Zero Overfitting Gap"
    )
else:
    print(
        "Variance Status       : "
        "Training-Test Gap Exists"
    )


[TASK 5 SUCCESS] Pruned Decision Tree Trained (max_depth=2).
Pruned Train Accuracy : 100.00%
Pruned Test Accuracy  : 100.00%
Pruned Overfitting Gap: 0.00%
Variance Status       : Zero Overfitting Gap


In [7]:
# ================================================================
# FINAL SUMMARY
# ================================================================

print(
    "\n========== "
    "API-DRIVEN DECISION TREE & OVERFITTING ENGINE "
    "=========="
)

print(
    "\nData Ingestion Status : "
    "REST API Ingestion Successful (HTTP 200 OK)"
)

print(
    f"Master Dataset Records : "
    f"{len(df)} Telemetry Logs"
)

print(
    "Features Included : "
    "4 Continuous Metrics "
    "(cpu_usage_pct, memory_usage_pct, "
    "disk_io_rate, network_latency_ms)"
)

print(
    "Target Output : "
    "Is_Failure "
    "(Binary Classification: "
    "0 = Normal, 1 = Failure)"
)

print("\nModel Training Metrics:")

print(
    f"- Stratified Train Split : "
    f"{len(X_train)} Records "
    f"({(y_train == 0).sum()} Normal / "
    f"{(y_train == 1).sum()} Failure)"
)

print(
    f"- Stratified Test Split : "
    f"{len(X_test)} Records "
    f"({(y_test == 0).sum()} Normal / "
    f"{(y_test == 1).sum()} Failure)"
)

print(
    f"- Unpruned Tree Accuracy : "
    f"Train = {train_accuracy * 100:.2f}% | "
    f"Test = {test_accuracy * 100:.2f}%"
)

print("\nFeature Importance Profiling:")

for _, row in importance_df.iterrows():

    print(
        f"- {row['Feature']} : "
        f"{row['Importance']:.4f}"
    )

print("\nHyperparameter Pruning & Regularization:")

print(
    f"- Baseline Unpruned Tree : "
    f"{test_accuracy * 100:.2f}% Test Accuracy"
)

print(
    f"- Pruned Tree (max_depth=2) : "
    f"{pruned_test_accuracy * 100:.2f}% Test Accuracy"
)

print("\nConclusion:")

print(
    "The pipeline dynamically fetches telemetry records "
    "through an HTTP REST endpoint, transforms the target "
    "into a binary classification problem, trains a Decision "
    "Tree, analyzes feature importance, and evaluates the "
    "effect of tree-depth regularization."
)


========== API-DRIVEN DECISION TREE & OVERFITTING ENGINE ==========

Data Ingestion Status : REST API Ingestion Successful (HTTP 200 OK)
Master Dataset Records : 150 Telemetry Logs
Features Included : 4 Continuous Metrics (cpu_usage_pct, memory_usage_pct, disk_io_rate, network_latency_ms)
Target Output : Is_Failure (Binary Classification: 0 = Normal, 1 = Failure)

Model Training Metrics:
- Stratified Train Split : 120 Records (40 Normal / 80 Failure)
- Stratified Test Split : 30 Records (10 Normal / 20 Failure)
- Unpruned Tree Accuracy : Train = 100.00% | Test = 100.00%

Feature Importance Profiling:
- disk_io_rate : 1.0000
- cpu_usage_pct : 0.0000
- memory_usage_pct : 0.0000
- network_latency_ms : 0.0000

Hyperparameter Pruning & Regularization:
- Baseline Unpruned Tree : 100.00% Test Accuracy
- Pruned Tree (max_depth=2) : 100.00% Test Accuracy

Conclusion:
The pipeline dynamically fetches telemetry records through an HTTP REST endpoint, transforms the target into a binary classifica